# SpectralEarth Visual Research Workbench Dashboard
## Interactive Exploration & Scientific Report Notebook

Welcome to the SpectralEarth Jupyter Research Playground. This notebook demonstrates how to build, layout, and visualize an integrated interactive frontend dashboard directly inside Jupyter using `ipywidgets` and `plotly` to explore reanalysis metrics, fit turbulence spectra, and export scientific reports.

### Outline:
1. **Import Libraries and Set Up Environment**
2. **Load and Prepare Reporting Data**
3. **Define Interactive Widgets**
4. **Create Dynamic Visualization Functions**
5. **Assemble the Integrated Frontend Layout**


### 1. Import Libraries and Set Up Environment
First, we import the core mathematical, interactive plotting, and UI layout libraries. We'll use `ipywidgets` for interactive controls and `plotly` for publication-ready responsive plots.


In [ ]:
# Standard analytical imports
import numpy as np
import pandas as pd
import torch

# Plotly visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Jupyter Widget and layouts
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout, Label, Play, jslink

print("Environment libraries successfully loaded.")


### 2. Load and Prepare Reporting Data
We'll generate a mock database representing operational metrics, including root-mean-squared-error (RMSE) trajectories over different forecast lead times, turbulent decay slope metrics ($\beta$), and boundary condition leakage rates for distinct regional configurations.


In [ ]:
# Seed random generators for consistency
np.random.seed(42)

# Generate a mock database table matching SQLite experiments and runs
runs_count = 120
lead_times = np.array([6, 12, 18, 24, 36, 48, 60, 72]) # hours

data_records = []
for run_idx in range(runs_count):
    # Choose random parameters
    transform_type = np.random.choice(["fft", "dct", "dwt", "dtcwt"])
    boundary_treatment = np.random.choice(["reflect", "periodic", "zero"])
    pad_width = np.random.choice([4, 8, 12])
    tukey_alpha = np.random.choice([0.0, 0.1, 0.2, 0.4])
    grid_size = np.random.choice([32, 64, 128])
    
    # Model spectral leakage and slope based on physics
    leakage = 1.85
    if boundary_treatment == "reflect":
        leakage -= 0.4
    if boundary_treatment == "periodic":
        leakage -= 0.6
    leakage -= (tukey_alpha * 1.5) # windowing suppresses leak
    leakage += np.random.normal(0, 0.05)
    leakage = max(0.1, leakage)
    
    # Model turbulent decay exponent (ideal is 3.0 or 5/3)
    fitted_beta = 3.0 + np.random.normal(0, 0.1)
    if transform_type == "fft" and boundary_treatment == "zero":
        fitted_beta -= 0.8 # boundary artifacts flattening the spectrum
    elif transform_type == "dwt":
        fitted_beta -= 0.4 # Haar wavelets are slightly lossy at fine modes
        
    for lt in lead_times:
        # Error grows over lead time
        growth_rate = 0.015 if transform_type in ["dtcwt", "dct"] else 0.024
        rmse = 0.05 + growth_rate * lt + (leakage * 0.02) + np.random.normal(0, 0.01)
        rmse = max(0.01, rmse)
        
        data_records.append({
            "run_id": f"run_{run_idx+1:03d}",
            "transform_type": transform_type,
            "boundary_treatment": boundary_treatment,
            "pad_width": int(pad_width),
            "tukey_alpha": float(tukey_alpha),
            "grid_size": int(grid_size),
            "spectral_leakage": float(leakage),
            "turbulent_slope_beta": float(fitted_beta),
            "lead_time": int(lt),
            "rmse": float(rmse),
            "ssim": float(max(0.2, 0.98 - rmse * 0.5))
        })

df_runs = pd.DataFrame(data_records)
print(f"Loaded database table of reporting metrics: {len(df_runs)} total rows across {runs_count} runs.")
df_runs.head(10)


### 3. Define Interactive Widgets
Now we'll define standard dropdown, slider, and selector widgets to allow interactive queries and coordinate filtering of the operational datasets.


In [ ]:
# Controls for filtering runs
widget_transform = widgets.Dropdown(
    options=['All', 'fft', 'dct', 'dwt', 'dtcwt'],
    value='All',
    description='Transform:',
    style={'description_width': 'initial'}
)

widget_boundary = widgets.Dropdown(
    options=['All', 'reflect', 'periodic', 'zero'],
    value='All',
    description='Boundary:',
    style={'description_width': 'initial'}
)

widget_grid_size = widgets.SelectionSlider(
    options=[32, 64, 128],
    value=64,
    description='Grid Size:',
    style={'description_width': 'initial'},
    continuous_update=False
)

widget_tukey_alpha = widgets.FloatSlider(
    value=0.1,
    min=0.0,
    max=0.4,
    step=0.1,
    description='Tukey Taper (\u03b1):',
    style={'description_width': 'initial'},
    continuous_update=False
)

# Out button to trigger report exporting
widget_export_btn = widgets.Button(
    description='Export Scientific Report',
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Saves a detailed markdown and statistical file from current selections',
    icon='file-code-o'
)

# Output log area
widget_output_log = widgets.Output()

print("Interactive widgets declared successfully.")


### 4. Create Dynamic Visualization Functions
We will write Python callback functions that hook into the widget state. Whenever a user moves a slider or toggles a dropdown, the function dynamically filters the DataFrame and updates:
1. An RMSE growth curve over forecast lead times.
2. A scatter plot mapping Spectral Leakage versus fitted Turbulent Slope ($\beta$).
3. Text metric cards containing average SSIM and R-squared statistics.


In [ ]:
# Dynamic Plotting Output Container
plot_output = widgets.Output()

def update_dashboard(*args):
    # Retrieve current filter values
    trans = widget_transform.value
    bound = widget_boundary.value
    grid = widget_grid_size.value
    alpha = widget_tukey_alpha.value
    
    # 1. Filter dataset
    filtered_df = df_runs.copy()
    if trans != 'All':
        filtered_df = filtered_df[filtered_df['transform_type'] == trans]
    if bound != 'All':
        filtered_df = filtered_df[filtered_df['boundary_treatment'] == bound]
    
    filtered_df = filtered_df[
        (filtered_df['grid_size'] == grid) & 
        (np.abs(filtered_df['tukey_alpha'] - alpha) < 0.05)
    ]
    
    # Clear old plots inside the output container
    with plot_output:
        plot_output.clear_output(wait=True)
        
        if len(filtered_df) == 0:
            print("No matching runs found for this configuration sweep.")
            return
            
        # 2. Create subplots: Lead-Time RMSE and Leakage vs turbulent slope scatter
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=(
                "Forecast Error (RMSE) over Lead Times",
                "Spectral Leakage vs fitted Turbulent Slope (\u03b2)"
            ),
            horizontal_spacing=0.15
        )
        
        # Plot 1: Mean RMSE growth over Lead Times
        grouped_rmse = filtered_df.groupby("lead_time")["rmse"].mean().reset_index()
        fig.add_trace(
            go.Scatter(
                x=grouped_rmse["lead_time"],
                y=grouped_rmse["rmse"],
                mode="lines+markers",
                name="Mean RMSE Trend",
                line=dict(color="#14b8a6", width=3),
                marker=dict(size=8, symbol="diamond")
            ),
            row=1, col=1
        )
        
        # Plot 2: Scatter of Leakage vs turbulent slope
        unique_runs = filtered_df.drop_duplicates(subset="run_id")
        fig.add_trace(
            go.Scatter(
                x=unique_runs["spectral_leakage"],
                y=unique_runs["turbulent_slope_beta"],
                mode="markers",
                name="Run Sweeps",
                marker=dict(
                    size=10,
                    color=unique_runs["ssim"],
                    colorscale="Viridis",
                    showscale=True,
                    colorbar=dict(
                        title="Mean SSIM",
                        x=1.05,
                        len=0.75
                    )
                ),
                text=unique_runs["run_id"]
            ),
            row=1, col=2
        )
        
        # Standard layout settings
        fig.update_layout(
            width=950,
            height=400,
            template="plotly_dark",
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0)',
            margin=dict(t=50, r=20, b=40, l=40),
            showlegend=False,
            font=dict(color="#94a3b8")
        )
        
        fig.update_xaxes(title_text="Forecast Lead Time (Hours)", row=1, col=1, gridcolor="#1e293b")
        fig.update_yaxes(title_text="Root-Mean-Squared-Error", row=1, col=1, gridcolor="#1e293b")
        fig.update_xaxes(title_text="Spectral Leakage Ratio", row=1, col=2, gridcolor="#1e293b")
        fig.update_yaxes(title_text="Turbulent Decay Slope \u03b2", row=1, col=2, gridcolor="#1e293b")
        
        # Display Plotly Chart
        import plotly.io as pio
        fig.show()
        
        # Print scientific summaries
        avg_rmse = filtered_df["rmse"].mean()
        avg_slope = filtered_df["turbulent_slope_beta"].mean()
        avg_leak = filtered_df["spectral_leakage"].mean()
        
        # Regime interpretation
        if abs(avg_slope - 3.0) < 0.2:
            regime = "Charney 2D Enstrophy Cascade (\u03b2 \u2248 3.0)"
        elif abs(avg_slope - 1.67) < 0.2:
            regime = "Kolmogorov 3D Kinetic Cascade (\u03b2 \u2248 5/3)"
        else:
            regime = f"Mixed/Transition Slope (\u03b2 = {avg_slope:.2f})"
            
        print(f"--- ACTIVE SWEEP DIAGNOSTIC SUMMARY ---")
        print(f"Matched Runs: {len(unique_runs)} sweeps | Total Lead-Time Steps: {len(filtered_df)}")
        print(f"Average RMSE: {avg_rmse:.4f} | Average Spectral Leakage: {avg_leak:.4f}")
        print(f"Estimated Turbulence Regime: {regime}")

# Attach callbacks to widget changes
widget_transform.observe(update_dashboard, 'value')
widget_boundary.observe(update_dashboard, 'value')
widget_grid_size.observe(update_dashboard, 'value')
widget_tukey_alpha.observe(update_dashboard, 'value')

print("Callback trigger functions declared.")


### 5. Assemble the Integrated Frontend Layout
We will organize our dropdowns, sliders, and buttons into an structured sidebar layout, and render it side-by-side with our plot output panel to build a dashboard inside Jupyter Notebook.


In [ ]:
# 1. Define Exporting callback to output scientific report from current selection
def handle_report_export(btn):
    with widget_output_log:
        widget_output_log.clear_output()
        print("Gathering active sweep parameters...")
        trans = widget_transform.value
        bound = widget_boundary.value
        grid = widget_grid_size.value
        alpha = widget_tukey_alpha.value
        
        # Sift records
        filtered_df = df_runs.copy()
        if trans != 'All':
            filtered_df = filtered_df[filtered_df['transform_type'] == trans]
        if bound != 'All':
            filtered_df = filtered_df[filtered_df['boundary_treatment'] == bound]
        
        filtered_df = filtered_df[
            (filtered_df['grid_size'] == grid) & 
            (np.abs(filtered_df['tukey_alpha'] - alpha) < 0.05)
        ]
        
        # Build file report
        report_path = "spectral_earth_active_report.md"
        with open(report_path, "w") as f:
            f.write("# SpectralEarth Research Workbench: Automated Export Report\n")
            f.write(f"- **Generated Date:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"- **Swept Transform:** {trans}\n")
            f.write(f"- **Boundary Treatment:** {bound}\n")
            f.write(f"- **Grid Resolution:** {grid}x{grid}\n")
            f.write(f"- **Windowing Tukey Alpha:** {alpha}\n\n")
            
            f.write("## 1. Aggregated Operational Performance\n")
            f.write(f"- **Matched Unique Run Sweeps:** {len(filtered_df.drop_duplicates('run_id'))}\n")
            f.write(f"- **Average Absolute RMSE:** {filtered_df['rmse'].mean():.5f}\n")
            f.write(f"- **Average Structural Similarity (SSIM):** {filtered_df['ssim'].mean():.4f}\n")
            f.write(f"- **Calculated Spectral Leakage Exponent:** {filtered_df['spectral_leakage'].mean():.4f}\n")
            f.write(f"- **Average Fitted Turbulent Slope (beta):** {filtered_df['turbulent_slope_beta'].mean():.4f}\n\n")
            
            f.write("## 2. Lead-Time Error Progression\n")
            f.write("| Forecast Lead Time (Hours) | Mean RMSE | Mean SSIM |\n")
            f.write("| --- | --- | --- |\n")
            for lt in sorted(filtered_df["lead_time"].unique()):
                sub = filtered_df[filtered_df["lead_time"] == lt]
                f.write(f"| {lt} | {sub['rmse'].mean():.5f} | {sub['ssim'].mean():.4f} |\n")
                
            f.write("\n\n*Report exported cleanly inside the active Jupyter environment. Ready for academic review.*")
            
        print(f"[+] Scientific report successfully written to '{report_path}'.")

widget_export_btn.on_click(handle_report_export)

# 2. Arrange layouts using HBox & VBox containers
controls_panel = VBox(
    [
        Label(value="DASHBOARD FILTERS", style=dict(font_weight='bold')),
        widget_transform,
        widget_boundary,
        widget_grid_size,
        widget_tukey_alpha,
        widgets.HTML("<hr>"),
        widget_export_btn,
        widget_output_log
    ],
    layout=Layout(
        width='250px',
        padding='15px',
        border='1px solid #1e293b',
        border_radius='8px',
        background_color='#0f172a'
    )
)

dashboard_layout = HBox(
    [controls_panel, plot_output],
    layout=Layout(
        gap='20px',
        padding='10px'
    )
)

# Initialize plots on load
update_dashboard()

# Display dashboard inside notebook
widgets.VBox([
    widgets.HTML("<h2>SpectralEarth Research Platform Workbench Controls</h2>"),
    dashboard_layout
])
